Setup

In [60]:
%pip install requests --quiet
from groq import Groq
from dotenv import load_dotenv
import os
import getpass

load_dotenv()

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL_NAME = "openai/gpt-oss-20b"


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Define Expert Configurations

In [61]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
        You are a Senior Technical Support Engineer.
        Be precise, structured, and code-focused.
        Provide debugging steps and sample fixes when applicable.
        """
    },
    "billing": {
        "system_prompt": """
        You are a Customer Billing Specialist.
        Be empathetic and professional.
        Reference subscription policies when needed.
        Offer clear next steps for refunds or disputes.
        """
    },
    "general": {
        "system_prompt": """
        You are a friendly Customer Support Assistant.
        Provide helpful and conversational responses.
        """
    },
    "tool": {
        "system_prompt": None  
    }
}

In [62]:
import requests

def get_bitcoin_price():
    try:
        url = "https://api.coingecko.com/api/v3/simple/price"
        params = {
            "ids": "bitcoin",
            "vs_currencies": "usd"
        }

        response = requests.get(url, params=params)
        data = response.json()

        price = data["bitcoin"]["usd"]

        return f"The current price of Bitcoin is ${price:,} USD."
    
    except Exception as e:
        return f"Sorry, I couldn't fetch the Bitcoin price right now. Error: {str(e)}"


Router Function (Core MoE Logic)

In [63]:
def route_prompt(user_input: str) -> str:
    router_prompt = f"""
    Classify the following customer query into ONE of these categories:
    [technical, billing, general, tool]

    Use "tool" if the query requires real-time data such as:
    - Cryptocurrency prices
    - Stock prices
    - Weather
    - Live information

    Return ONLY the category name. No explanation.

    Query:
    {user_input}
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are an intent classification system."},
            {"role": "user", "content": router_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()

    if category not in MODEL_CONFIG:
        category = "general"

    return category

Orchestrator Function

In [64]:
def process_request(user_input: str) -> str:
    category = route_prompt(user_input)

    if category == "tool":
        tool_response = get_bitcoin_price()
        return f"[ROUTED TO: TOOL]\n\n{tool_response}"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return f"[ROUTED TO: {category.upper()} EXPERT]\n\n{response.choices[0].message.content.strip()}"

Testing the System

In [65]:
print(process_request("My python script is throwing an IndexError on line 5."))

[ROUTED TO: TECHNICAL EXPERT]

## Quick‑look Checklist (what to verify on **line 5**)

| # | What to check | Why it matters | Typical fix |
|---|---------------|----------------|-------------|
| 1 | **Which variable is indexed?** | The `IndexError` is always about a list/tuple/string/slice. | Pinpoint the variable (`mylist`, `data`, etc.). |
| 2 | **What index is used?** | If the index is a literal (`5`), a loop variable (`i+1`), or a function return, you need to know its range. | Make sure the index is `< len(variable)`. |
| 3 | **What is the length of the variable?** | If the list is empty or shorter than expected, any positive index fails. | Print `len(variable)` before the offending line. |
| 4 | **Is the variable being modified earlier?** | A previous slice or delete could shorten the list. | Re‑evaluate the state of the variable just before line 5. |
| 5 | **Is the code inside a loop?** | Off‑by‑one errors often happen in loops (`for i in range(len(lst)):` then `lst[i+1]`). | Cha

In [66]:
print(process_request("I was charged twice for my subscription this month."))

[ROUTED TO: BILLING EXPERT]

I’m really sorry you’ve been charged twice for your subscription this month – that’s definitely not what we want for our customers.

**What’s likely happening**

Sometimes a duplicate charge shows up when a payment processor retries a failed transaction. It’s an error on our end, not a second subscription you’re being billed for.

**Next steps**

1. **Check the details** – If you can, please send me the transaction IDs (or a screenshot of the bank statement showing the two charges).  
2. **We’ll investigate** – I’ll pull up your account and compare the dates, amounts, and payment method to confirm the duplicate.  
3. **Refund** – Once confirmed, we’ll issue a refund for the duplicate charge. Our policy states that refunds are processed within 5–7 business days and will appear on the original payment method.  
4. **Confirm cancellation** – If the second charge is for a new subscription period, we’ll cancel that to prevent future billing.

**If you’d rather n

In [67]:
print(process_request("Do you offer student discounts?"))

[ROUTED TO: BILLING EXPERT]

Hi there! 👋

I’m glad you asked about student discounts. We do offer a special rate for students, and I’d be happy to help you get set up.

**What we need to confirm:**
1. **Student status** – We’ll need a valid student ID, a recent transcript, or a .edu email address to verify your enrollment.
2. **Current subscription plan** – If you’re already subscribed, we can apply the discount to your next billing cycle. If you’re just starting, we can activate the student plan immediately.

**How to get started:**
1. **Send a photo or scan** of your student ID or a recent transcript to our support mailbox at studentdiscounts@ourcompany.com.  
2. **Include your account email** (or the email you used to sign up) so we can locate your account quickly.
3. **Optional**: If you have a .edu email address, just let us know and we’ll verify it in our system.

Once we receive your verification, we’ll apply the discount and you’ll see the reduced rate on your next invoice. If 

In [68]:
print(process_request("What is the current price of Bitcoin?"))

[ROUTED TO: TOOL]

The current price of Bitcoin is $68,892 USD.
